<a href="https://colab.research.google.com/github/ishuu9837/Fine-tuning-Llama-3.2/blob/main/Fine_Tuning_Llama_3_2_for_Domain_Specific_(Finance)_Question_Answering_using_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets evaluate rouge_score sacrebleu

In [2]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('collab'))

In [3]:
!pip uninstall -y pyarrow datasets
!pip install -U pyarrow datasets

Found existing installation: pyarrow 25.0.0
Uninstalling pyarrow-25.0.0:
  Successfully uninstalled pyarrow-25.0.0
Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
  Using cached pyarrow-25.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
Using cached pyarrow-25.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (50.1 MB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)


In [4]:
from datasets import load_dataset

dataset = load_dataset("gbharti/finance-alpaca", split="train")
print(dataset)
print(dataset[0])

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 68912
})
{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0%

In [5]:
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    user_msg = example["instruction"]
    if example["input"].strip():
        user_msg += "\n\n" + example["input"]

    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 Jul 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

For a car, what scams can be plotted with 0% financing vs rebate?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, 

In [6]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]
print(f"Train = {len(train_dataset)}")
print(f"Eval  = {len(eval_dataset)}")

Train = 62020
Eval  = 6892


In [7]:
train_dataset = train_dataset.shuffle(seed=42).select(range(3000))
eval_dataset = eval_dataset.shuffle(seed=42).select(range(200))
print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 3000
})
Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 200
})


In [8]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True Tesla T4


In [9]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
print("Model loaded successfully!")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded successfully!


In [10]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [11]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama3-finance-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=11,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    report_to="none",
)

In [15]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    **training_args.to_dict(),
    max_length=512,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

... (3 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.757243,1.413878,1.469830,57383.000000,0.672712
100,1.638900,1.371985,1.360849,116086.000000,0.677147
150,1.787566,1.360844,1.406105,174566.000000,0.678918
200,1.654339,1.352614,1.349020,235585.000000,0.680122
250,1.634044,1.345859,1.346128,295322.000000,0.681395
300,1.548943,1.341286,1.344509,354128.000000,0.682696
350,1.544022,1.340692,1.343290,411239.000000,0.682765
375,1.449047,1.340816,1.343707,438442.000000,0.682911


TrainOutput(global_step=375, training_loss=1.6242608184814453, metrics={'train_runtime': 7492.1945, 'train_samples_per_second': 0.4, 'train_steps_per_second': 0.05, 'total_flos': 1.0055498233356288e+16, 'train_loss': 1.6242608184814453, 'epoch': 1.0})

In [17]:
trainer.save_model("./llama3-finance-lora-final")
tokenizer.save_pretrained("./llama3-finance-lora-final")

('./llama3-finance-lora-final/tokenizer_config.json',
 './llama3-finance-lora-final/chat_template.jinja',
 './llama3-finance-lora-final/tokenizer.json')

In [18]:
import torch

model.eval()

def generate_answer(question, max_new_tokens=150):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Reload raw eval data (with original instruction/output columns) for reference text
raw_eval = dataset["test"].shuffle(seed=42).select(range(50))

predictions = []
references = []

for i, example in enumerate(raw_eval):
    question = example["instruction"]
    if example["input"].strip():
        question += "\n\n" + example["input"]
    reference = example["output"]

    pred = generate_answer(question)
    predictions.append(pred)
    references.append(reference)

    if i % 10 == 0:
        print(f"[{i}/50] Done")

print("Generation complete.")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[0/50] Done
[10/50] Done
[20/50] Done
[30/50] Done
[40/50] Done
Generation complete.


In [19]:
print(dataset["test"][0])

{'instruction': 'Expiring 401(k) Stock Option and Liquidation Implications', 'input': '', 'output': "I have had this happen a couple of times because of splits or sales of portions of the company.  The general timeline was to announce how the split was to be handled; then the split; then a freeze in purchasing stock in the other company; then a freeze in sales; followed by a short blackout period; then the final transfers to funds/options/cash based on a mapping announced at the start of the process. You need to answer two questions: To determine if the final transactions will make the market move you have to understand how many shares are involved compared to the typical daily volume. There are two caveats: professional investors will be aware of the transaction date and can either ignore the employee transactions or try and take advantage of them; There may also be a mirroring set of transactions if the people left in the old company were awarded shares in your company as part of the

In [20]:
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

bleu_results = bleu.compute(predictions=predictions, references=[[r] for r in references])
rouge_results = rouge.compute(predictions=predictions, references=references)

print("=== BLEU ===")
print(f"BLEU score: {bleu_results['bleu']:.4f}")

print("\n=== ROUGE ===")
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

=== BLEU ===
BLEU score: 0.0733

=== ROUGE ===
rouge1: 0.4018
rouge2: 0.1774
rougeL: 0.3372
rougeLsum: 0.3537


In [ ]:
# Removed an embedded credential-like value. Use a secrets manager for credentials.

In [21]:
import math

def compute_perplexity(model, tokenizer, texts, max_length=512):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
            outputs = model(**inputs, labels=inputs["input_ids"])
            # outputs.loss is average cross-entropy loss per token
            num_tokens = inputs["input_ids"].shape[1]
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity

# Use the same 50 formatted (chat-templated) eval examples for consistency
eval_texts = dataset["test"].shuffle(seed=42).select(range(50))["text"]

ppl = compute_perplexity(model, tokenizer, eval_texts)
print(f"Perplexity: {ppl:.4f}")

Perplexity: 3.9267


In [22]:
# Load base model fresh (no LoRA adapter)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.eval()

# Perplexity on same eval texts
base_ppl = compute_perplexity(base_model, tokenizer, eval_texts)
print(f"Base model Perplexity: {base_ppl:.4f}")
print(f"Fine-tuned Perplexity: {ppl:.4f}")
print(f"Improvement: {((base_ppl - ppl) / base_ppl * 100):.2f}%")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model Perplexity: 30.5891
Fine-tuned Perplexity: 3.9267
Improvement: 87.16%


In [23]:
for i in range(3):
    print(f"--- Example {i+1} ---")
    print(f"Q: {raw_eval[i]['instruction']}")
    print(f"\nModel answer:\n{predictions[i]}")
    print(f"\nReference answer:\n{references[i]}")
    print("\n" + "="*80 + "\n")

--- Example 1 ---
Q: Pose a hypothetical question about automation.

Model answer:
What are the potential benefits and drawbacks of automating certain tasks?

Reference answer:
What would be the implications if machines replaced all manual labour?


--- Example 2 ---
Q: Given the word, list 3 synonyms.

Model answer:
large, enormous, gigantic

Reference answer:
large, vast, immense


--- Example 3 ---
Q: Connect a device to a local network.

Model answer:
To connect the Android tablet to a local network, the tablet needs to be connected to a router. The tablet can be connected to the router via Wi-Fi or Ethernet. Once connected, the tablet can access the internet and other devices on the network.

Reference answer:
To connect the Android tablet to the local network, you will need to open the device's Wi-Fi settings. Then select the local Wi-Fi network and enter the required credentials (if any) to connect to the network. Once connected, the device should be able to access the local net

In [25]:
!zip -r llama3-finance-lora-final.zip ./llama3-finance-lora-final
from google.colab import files
files.download('llama3-finance-lora-final.zip')

  adding: llama3-finance-lora-final/ (stored 0%)
  adding: llama3-finance-lora-final/README.md (deflated 65%)
  adding: llama3-finance-lora-final/adapter_model.safetensors (deflated 21%)
  adding: llama3-finance-lora-final/adapter_config.json (deflated 59%)
  adding: llama3-finance-lora-final/tokenizer_config.json (deflated 44%)
  adding: llama3-finance-lora-final/tokenizer.json (deflated 85%)
  adding: llama3-finance-lora-final/chat_template.jinja (deflated 71%)
  adding: llama3-finance-lora-final/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>